In [6]:
# ============================================================
# INDIAN QUANT PORTFOLIO & RISK ENGINE
# STEP 2: RISK & DEPENDENCE ANALYSIS
# ============================================================

import os
import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

DATA_DIR = "data/raw"
OUTPUT_DIR = "data/processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRADING_DAYS = 252
CONFIDENCE_LEVEL = 0.95


# ============================================================
# LOAD DATA
# ============================================================

prices = pd.read_csv(
    os.path.join(DATA_DIR, "adjusted_close_prices.csv"),
    index_col="Date",
    parse_dates=True
)

returns = pd.read_csv(
    os.path.join(DATA_DIR, "daily_returns.csv"),
    index_col="Date",
    parse_dates=True
)

market = pd.read_csv(
    os.path.join(DATA_DIR, "market_prices.csv"),
    index_col="Date",
    parse_dates=True
)


prices = prices.sort_index()
returns = returns.sort_index()
market = market.sort_index()

# Remove rows where everything is missing
returns = returns.dropna(how="all")
prices = prices.dropna(how="all")


print("=" * 70)
print("RISK & DEPENDENCE ANALYSIS")
print("=" * 70)

print(f"\nStocks: {len(returns.columns)}")
print(f"Observations: {len(returns)}")
print(f"Period: {returns.index.min().date()} → {returns.index.max().date()}")


# ============================================================
# 1. CLEAN RETURNS
# ============================================================

# Remove impossible/infinite values
returns = returns.replace(
    [np.inf, -np.inf],
    np.nan
)

# Don't blindly fill return data.
# Missing return = no observation.
returns = returns.dropna(
    how="all"
)


# ============================================================
# 2. DAILY RETURNS
# ============================================================

daily_returns = returns.copy()


# ============================================================
# 3. WEEKLY RETURNS
# ============================================================

weekly_returns = (
    prices
    .resample("W-FRI")
    .last()
    .pct_change(fill_method=None)
)


# ============================================================
# 4. MONTHLY RETURNS
# ============================================================

monthly_returns = (
    prices
    .resample("ME")
    .last()
    .pct_change(fill_method=None)
)


# ============================================================
# 5. ANNUALIZED RETURN
# ============================================================

annualized_return = (
    daily_returns.mean() * TRADING_DAYS
)


# ============================================================
# 6. ANNUALIZED VOLATILITY
# ============================================================

annualized_volatility = (
    daily_returns.std() * np.sqrt(TRADING_DAYS)
)


# ============================================================
# 7. DOWNSIDE VOLATILITY
# ============================================================

# Use zero as minimum acceptable return.
downside_returns = daily_returns.clip(upper=0)

downside_volatility = (
    downside_returns.std() * np.sqrt(TRADING_DAYS)
)


# ============================================================
# 8. SKEWNESS & KURTOSIS
# ============================================================

skewness = daily_returns.skew()

kurtosis = daily_returns.kurtosis()


# ============================================================
# 9. CUMULATIVE RETURNS
# ============================================================

cumulative_returns = (
    1 + daily_returns.fillna(0)
).cumprod()


# ============================================================
# 10. MAXIMUM DRAWDOWN
# ============================================================

running_max = cumulative_returns.cummax()

drawdown = (
    cumulative_returns / running_max
) - 1

maximum_drawdown = drawdown.min()


# ============================================================
# 11. DRAWDOWN DURATION
# ============================================================

drawdown_duration = {}

for stock in cumulative_returns.columns:

    series = cumulative_returns[stock]

    peak = series.cummax()

    underwater = series < peak

    groups = (
        underwater
        .ne(underwater.shift())
        .cumsum()
    )

    durations = (
        underwater
        .groupby(groups)
        .sum()
    )

    drawdown_duration[stock] = durations.max()


# ============================================================
# 12. BEST / WORST DAY
# ============================================================

best_day = daily_returns.max()

worst_day = daily_returns.min()


# ============================================================
# 13. HISTORICAL VaR
# ============================================================

historical_var = daily_returns.quantile(
    1 - CONFIDENCE_LEVEL
)


# ============================================================
# 14. HISTORICAL CVaR / EXPECTED SHORTFALL
# ============================================================

def calculate_cvar(series, confidence=0.95):

    var = series.quantile(1 - confidence)

    tail_losses = series[
        series <= var
    ]

    return tail_losses.mean()


historical_cvar = daily_returns.apply(
    calculate_cvar,
    confidence=CONFIDENCE_LEVEL
)


# ============================================================
# 15. PARAMETRIC VaR
# ============================================================

z_score = 1.645  # 95% one-tailed

parametric_var = (
    daily_returns.mean()
    - z_score * daily_returns.std()
)


# ============================================================
# 16. BETA AGAINST NIFTY 50
# ============================================================

if "^NSEI" in market.columns:

    nifty_returns = market["^NSEI"].pct_change()

    aligned = pd.concat(
        [daily_returns, nifty_returns.rename("NIFTY")],
        axis=1
    ).dropna()

    beta = {}

    nifty_variance = aligned["NIFTY"].var()

    for stock in daily_returns.columns:

        covariance = aligned[stock].cov(
            aligned["NIFTY"]
        )

        beta[stock] = (
            covariance / nifty_variance
        )

    beta = pd.Series(beta)

else:

    print("\nWARNING: NIFTY 50 data unavailable.")
    beta = pd.Series(
        np.nan,
        index=daily_returns.columns
    )


# ============================================================
# 17. SHARPE RATIO
# ============================================================

# For initial analysis we use 0 as risk-free rate.
# Later we can add a proper Indian risk-free series.

sharpe_ratio = (
    annualized_return /
    annualized_volatility
)


# ============================================================
# 18. SORTINO RATIO
# ============================================================

sortino_ratio = (
    annualized_return /
    downside_volatility
)


# ============================================================
# 19. CALMAR RATIO
# ============================================================

calmar_ratio = (
    annualized_return /
    abs(maximum_drawdown)
)


# ============================================================
# 20. COMPLETE RISK SUMMARY
# ============================================================

risk_summary = pd.DataFrame({

    "Annualized_Return": annualized_return,

    "Annualized_Volatility":
        annualized_volatility,

    "Downside_Volatility":
        downside_volatility,

    "Sharpe_Ratio":
        sharpe_ratio,

    "Sortino_Ratio":
        sortino_ratio,

    "Calmar_Ratio":
        calmar_ratio,

    "Maximum_Drawdown":
        maximum_drawdown,

    "Max_Drawdown_Duration_Days":
        pd.Series(drawdown_duration),

    "Historical_VaR_95":
        historical_var,

    "Historical_CVaR_95":
        historical_cvar,

    "Parametric_VaR_95":
        parametric_var,

    "Beta_NIFTY":
        beta,

    "Skewness":
        skewness,

    "Kurtosis":
        kurtosis,

    "Best_Day":
        best_day,

    "Worst_Day":
        worst_day
})


# ============================================================
# 21. CORRELATION MATRIX
# ============================================================

correlation_matrix = daily_returns.corr()


# ============================================================
# 22. COVARIANCE MATRIX
# ============================================================

covariance_matrix = daily_returns.cov()


# Annualized covariance
annualized_covariance = (
    covariance_matrix * TRADING_DAYS
)


# ============================================================
# 23. EWMA COVARIANCE MATRIX
# ============================================================

# More recent observations receive higher weights.

LAMBDA = 0.94


def ewma_covariance(data, decay=0.94):

    # Keep only rows where all stocks have observations
    data = data.dropna(how="any")

    X = data.to_numpy(dtype=float)

    n = len(X)

    if n == 0:
        raise ValueError("No complete observations available for EWMA covariance.")

    # More recent observations get larger weights
    weights = np.array([
        (1 - decay) * decay ** i
        for i in range(n)
    ])

    # Oldest -> newest
    weights = weights[::-1]

    weights = weights / weights.sum()

    # Weighted mean
    mean = np.average(
        X,
        axis=0,
        weights=weights
    )

    centered = X - mean

    # Weighted covariance
    covariance = (
        centered.T
        @ (centered * weights[:, None])
    )

    return pd.DataFrame(
        covariance,
        index=data.columns,
        columns=data.columns
    )


RISK & DEPENDENCE ANALYSIS

Stocks: 15
Observations: 2874
Period: 2015-01-02 → 2026-08-19


/var/folders/z8/txw1dwts4zs619_720bsj9ww0000gn/T/ipykernel_8445/1684790004.py:261: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  aligned = pd.concat(


In [7]:
ewma_cov = ewma_covariance(
    daily_returns,
    decay=LAMBDA
)

ewma_cov_annualized = (
    ewma_cov * TRADING_DAYS
)

In [9]:

# ============================================================
# 24. ROLLING VOLATILITY
# ============================================================

rolling_volatility = (
    daily_returns
    .rolling(60)
    .std()
    * np.sqrt(TRADING_DAYS)
)


# ============================================================
# 25. ROLLING BETA
# ============================================================

rolling_beta = pd.DataFrame(
    index=daily_returns.index,
    columns=daily_returns.columns,
    dtype=float
)

if "^NSEI" in market.columns:

    nifty_returns = market["^NSEI"].pct_change()

    for stock in daily_returns.columns:

        covariance = (
            daily_returns[stock]
            .rolling(60)
            .cov(nifty_returns)
        )

        nifty_variance = (
            nifty_returns
            .rolling(60)
            .var()
        )

        rolling_beta[stock] = (
            covariance / nifty_variance
        )


# ============================================================
# 26. ROLLING SHARPE
# ============================================================

rolling_mean = (
    daily_returns
    .rolling(60)
    .mean()
)

rolling_std = (
    daily_returns
    .rolling(60)
    .std()
)

rolling_sharpe = (
    rolling_mean / rolling_std
) * np.sqrt(TRADING_DAYS)


# ============================================================
# 27. SAVE OUTPUTS
# ============================================================

risk_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "risk_summary.csv"
    )
)

correlation_matrix.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "correlation_matrix.csv"
    )
)

covariance_matrix.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "daily_covariance_matrix.csv"
    )
)

annualized_covariance.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "annualized_covariance_matrix.csv"
    )
)

ewma_cov_annualized.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ewma_covariance_matrix.csv"
    )
)

rolling_volatility.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "rolling_volatility_60d.csv"
    )
)

rolling_beta.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "rolling_beta_60d.csv"
    )
)

rolling_sharpe.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "rolling_sharpe_60d.csv"
    )
)

cumulative_returns.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "cumulative_returns.csv"
    )
)

drawdown.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "drawdown.csv"
    )
)

weekly_returns.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "weekly_returns.csv"
    )
)

monthly_returns.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "monthly_returns.csv"
    )
)


# ============================================================
# 28. PRINT RANKINGS
# ============================================================

print("\n" + "=" * 70)
print("TOP STOCKS BY SHARPE RATIO")
print("=" * 70)

print(
    risk_summary[
        [
            "Annualized_Return",
            "Annualized_Volatility",
            "Sharpe_Ratio",
            "Sortino_Ratio",
            "Maximum_Drawdown",
            "Beta_NIFTY"
        ]
    ]
    .sort_values(
        "Sharpe_Ratio",
        ascending=False
    )
    .round(4)
    .head(15)
)


print("\n" + "=" * 70)
print("HIGHEST VOLATILITY")
print("=" * 70)

print(
    risk_summary[
        "Annualized_Volatility"
    ]
    .sort_values(
        ascending=False
    )
    .head(10)
    .round(4)
)


print("\n" + "=" * 70)
print("WORST MAXIMUM DRAWDOWNS")
print("=" * 70)

print(
    risk_summary[
        "Maximum_Drawdown"
    ]
    .sort_values()
    .head(10)
    .round(4)
)


print("\n" + "=" * 70)
print("FILES CREATED")
print("=" * 70)

for file in sorted(os.listdir(OUTPUT_DIR)):
    print(
        os.path.join(
            OUTPUT_DIR,
            file
        )
    )

print("\nRisk analysis completed successfully.")


TOP STOCKS BY SHARPE RATIO
               Annualized_Return  Annualized_Volatility  Sharpe_Ratio  \
BRITANNIA.NS              0.1989                 0.2396        0.8301   
NESTLEIND.NS              0.1724                 0.2279        0.7566   
APOLLOHOSP.NS             0.2309                 0.3093        0.7466   
BHARTIARTL.NS             0.2031                 0.2887        0.7036   
TATASTEEL.NS              0.2276                 0.3561        0.6392   
NTPC.NS                   0.1600                 0.2588        0.6180   
DLF.NS                    0.2389                 0.4223        0.5658   
TECHM.NS                  0.1535                 0.2924        0.5250   
KOTAKBANK.NS              0.1336                 0.2591        0.5157   
INFY.NS                   0.1356                 0.2681        0.5057   
SUNPHARMA.NS              0.1212                 0.2816        0.4304   
WIPRO.NS                  0.0933                 0.2569        0.3632   
DRREDDY.NS             